## Setup and Imports

In [57]:
import gsops.backend as gs
import polyscope as ps

from geomfum.convert import NeighborFinder
from geomfum.dataset import NotebooksDataset
from geomfum.shape import TriangleMesh
from geomfum.shape.hierarchical import HierarchicalMesh


# Fast Processing with Hierarchical Meshes

This notebook demonstrates how to efficiently compute geometric quantities on high-resolution meshes by using hierarchical mesh representations. The key idea is to:

1. Create a low-resolution version of a high-resolution mesh
2. Compute expensive operations (eigenvectors, geodesic distances, etc.) on the low-resolution mesh
3. Efficiently extend these quantities back to the high-resolution mesh using barycentric interpolation

This approach provides significant computational savings while maintaining good approximation quality.

In [58]:
dataset = NotebooksDataset()

mesh_a = TriangleMesh.from_file(dataset.get_filename("cat-00"))

(mesh_a.n_vertices, mesh_a.n_faces)

(7207, 14410)

## Step 1: Create Hierarchical Mesh

We'll create a hierarchical mesh representation with a low-resolution version containing approximately 500 vertices. This downsampled mesh will be used for expensive computations, which are then efficiently transferred to the high-resolution mesh.

In [59]:
# Target number of vertices for the low-resolution mesh
NUMBER_OF_SAMPLES = 500
hmesh_a = HierarchicalMesh.from_registry(mesh_a, min_n_samples=NUMBER_OF_SAMPLES)

print(
    f"High-res mesh: {hmesh_a.high.n_vertices} vertices, {hmesh_a.high.n_faces} faces"
)
print(f"Low-res mesh:  {hmesh_a.low.n_vertices} vertices, {hmesh_a.low.n_faces} faces")

# Visualize both resolutions side by side
ps.init()
ps_mesh_a = ps.register_surface_mesh(
    "Cat High Res", hmesh_a.high.vertices, hmesh_a.high.faces
)
ps_mesh_b = ps.register_surface_mesh(
    "Cat Low Res", hmesh_a.low.vertices + gs.array([0.3, 0, 0]), hmesh_a.low.faces
)
ps.show()
ps.remove_all_structures()

High-res mesh: 7207 vertices, 14410 faces
Low-res mesh:  502 vertices, 1000 faces


Having the two resolutions, we can compute quantities on the low resolution mesh and reproject them in the high resolution mesh.

## Step 2: Extending the Laplacian Basis

We can compute the Laplacian basis on the low-resolution mesh and then efficiently extend it to the high-resolution mesh using barycentric interpolation.

In [62]:
hmesh_a.low.laplacian.find_spectrum(spectrum_size=5, set_as_basis=True)
ps_mesh_b = ps.register_surface_mesh(
    "Cat Low Res", hmesh_a.low.vertices, hmesh_a.low.faces
)
ps_mesh_b.add_scalar_quantity(
    "LB Spectrum First Eigenvector",
    hmesh_a.low.basis.vecs[:, 1],
    enabled=True,
    defined_on="vertices",
    cmap="coolwarm",
)
ps.show()


ps.remove_all_structures()

In [63]:
hmesh_a.extend_basis()
ps_mesh_b = ps.register_surface_mesh(
    "Cat High Res", hmesh_a.high.vertices, hmesh_a.high.faces
)
ps_mesh_b.add_scalar_quantity(
    "LB Spectrum First Eigenvector",
    hmesh_a.high.basis.vecs[:, 1],
    enabled=True,
    defined_on="vertices",
    cmap="coolwarm",
)
ps.show()

The `extend_basis()` method is a convenience function that extends the basis from low to high resolution. More generally, we can extend any scalar function using `scalar_low_high()`:

In [64]:
hmesh_a.scalar_low_high(hmesh_a.low.basis.vecs[:, 1])

array([1.69095187, 1.76427966, 1.75787054, ..., 5.28022389, 5.28585411,
       1.69439715], shape=(7207,))

In [66]:
# the core part of the code is given by the baricentric map
bary_map = hmesh_a._baryc_map
print(bary_map.shape)
basis_high = bary_map @ hmesh_a.low.basis.vecs

(7207, 502)


### The Barycentric Map

The core mechanism for extending quantities from low to high resolution is the **barycentric map**. This sparse matrix encodes barycentric coordinates, allowing efficient interpolation of any quantity from the low-resolution mesh to the high-resolution mesh.

## Step 3: Extending the Laplacian Operator

We can extend the Laplacian operator itself from low to high resolution using the formula: `L_high = B @ L_low @ B^T`, where `B` is the barycentric map.

In [67]:
L, M = hmesh_a.low.laplacian.stiffness_matrix, hmesh_a.high.laplacian.mass_matrix

L.shape

(502, 502)

In [68]:
L_high = bary_map @ L @ bary_map.T

L_high.shape

(7207, 7207)

## Step 4: Computing Geodesic Distances

We can compute geodesic distances on the low-resolution mesh and extend them to the high-resolution mesh using the same barycentric interpolation approach.

In [69]:
from geomfum.metric.mesh import ScipyGraphShortestPathMetric

metric = ScipyGraphShortestPathMetric(hmesh_a.low)
dists_low = metric.dist_matrix()
dists_low.shape

(502, 502)

In [70]:
dists_high = bary_map @ dists_low @ bary_map.T
dists_high.shape

(7207, 7207)

In [71]:
# visualization of the distance from a point
ps.remove_all_structures()
ps_mesh_b = ps.register_surface_mesh(
    "Cat Low Res", hmesh_a.low.vertices, hmesh_a.low.faces
)
ps.register_point_cloud("Origin_Low", hmesh_a.low.vertices[0:1], radius=0.02)
ps_mesh_b.add_scalar_quantity(
    "Distance from Vertex 0",
    dists_low[0],
    enabled=True,
    defined_on="vertices",
    cmap="reds",
)


ps.show()
ps.remove_all_structures()

In [72]:
# Visualization of the distance from a point on the high-resolution mesh
knn = NeighborFinder()
high_index = knn(hmesh_a.low.vertices[0:1], hmesh_a.high.vertices).flatten()
ps_mesh_b = ps.register_surface_mesh(
    "Cat High Res", hmesh_a.high.vertices, hmesh_a.high.faces
)
ps.register_point_cloud("Origin_High", hmesh_a.high.vertices[high_index], radius=0.02)

ps_mesh_b.add_scalar_quantity(
    "Distance from Vertex 0",
    dists_high[high_index].T.flatten(),
    enabled=True,
    defined_on="vertices",
    cmap="reds",
)
ps.show()
ps.remove_all_structures()

## Summary

This notebook demonstrated the hierarchical mesh workflow for efficient processing of high-resolution meshes:

- **Hierarchical Mesh Creation**: Downsampling high-resolution meshes to a manageable size
- **Efficient Computation**: Computing expensive operations (eigenvectors, geodesic distances) on low-resolution meshes
- **Barycentric Interpolation**: Extending results back to high-resolution meshes via the barycentric map

This approach enables working with detailed meshes while keeping computation costs reasonable, making it ideal for:
- Interactive applications
- Large-scale mesh processing
- Prototyping geometric algorithms

The key insight is that many geometric quantities vary smoothly across the mesh surface, making them amenable to interpolation from coarse to fine representations.